In [2]:
import os, time, cv2, torch, random, numpy as np
from PIL import Image
from ultralytics import YOLO
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn,
    maskrcnn_resnet50_fpn
)
from torchvision.transforms import functional as F

# ==============================
# PATHS  (ONLY sampled_dataset)
# ==============================
IMG_DIR = r"C:\Users\Adithya R\CV Hackathon\sampled_dataset\sampled_dataset\images"
ANN_DIR = r"C:\Users\Adithya R\CV Hackathon\sampled_dataset\sampled_dataset\annotations"
OUT_DIR = r"C:\Users\Adithya R\CV Hackathon\outputs"

os.makedirs(OUT_DIR, exist_ok=True)

CONF = 0.5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ==============================
# USE ONLY EXISTING SAMPLED DATA
# ==============================
TEST_IMAGES = [f for f in os.listdir(IMG_DIR)
               if os.path.exists(os.path.join(ANN_DIR, f.replace(".jpg",".txt")))]

print(f"Running benchmark on sampled_dataset → {len(TEST_IMAGES)} images")

# ==============================
# LOAD VISDRONE GT
# ==============================
def load_visdrone_boxes(txt_path):
    boxes = []
    with open(txt_path) as f:
        for line in f:
            x,y,w,h,score,cls,trunc,occ = map(int, line.strip().split(","))
            if w>0 and h>0:
                boxes.append([x, y, x+w, y+h])
    return np.array(boxes)

# ==============================
# IOU FUNCTION
# ==============================
def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter = max(0, xB-xA) * max(0, yB-yA)
    areaA = (boxA[2]-boxA[0])*(boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0])*(boxB[3]-boxB[1])
    union = areaA + areaB - inter
    return inter / max(union, 1e-6)

def is_small(box, img_area, thresh=0.01):
    w = box[2]-box[0]
    h = box[3]-box[1]
    return (w*h) < (img_area * thresh)

# ==============================
# METRIC CALCULATION
# ==============================
def evaluate(pred_boxes, gt_boxes, img_area):
    tp, fp = 0, 0
    used = set()
    ious = []

    for p in pred_boxes:
        best_iou = 0
        best_j = -1
        for j,g in enumerate(gt_boxes):
            if j in used: continue
            score = iou(p, g)
            if score > best_iou:
                best_iou = score
                best_j = j

        if best_iou >= 0.5:
            tp += 1
            used.add(best_j)
            ious.append(best_iou)
        else:
            fp += 1

    fn = len(gt_boxes) - tp

    precision = tp / max(tp+fp, 1)
    recall = tp / max(tp+fn, 1)
    mean_iou = np.mean(ious) if len(ious)>0 else 0

    small_gt = [b for b in gt_boxes if is_small(b, img_area)]
    small_tp = sum(1 for b in small_gt if any(iou(b,p)>=0.5 for p in pred_boxes))
    small_recall = small_tp / max(len(small_gt),1)

    return {
        "precision": precision,
        "recall": recall,
        "mean_iou": mean_iou,
        "small_recall": small_recall
    }

# ==============================
# LOAD MODELS (NO SAM)
# ==============================
print("\nLoading models...\n")

frcnn = fasterrcnn_resnet50_fpn(weights="DEFAULT").to(DEVICE).eval()
mrcnn = maskrcnn_resnet50_fpn(weights="DEFAULT").to(DEVICE).eval()
rfdetr = YOLO("rtdetr-l.pt")

# ==============================
# RUN BENCHMARK
# ==============================

results_summary = {
    "faster_rcnn": [],
    "mask_rcnn": [],
    "rfdetr": []
}

for idx, img_name in enumerate(TEST_IMAGES):

    print(f"\nProcessing {idx+1}/{len(TEST_IMAGES)} : {img_name}")

    img_path = os.path.join(IMG_DIR, img_name)
    ann_path = os.path.join(ANN_DIR, img_name.replace(".jpg",".txt"))

    pil = Image.open(img_path).convert("RGB")
    img = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
    h,w,_ = img.shape
    img_area = h*w

    gt = load_visdrone_boxes(ann_path)

    # ---------- Faster R-CNN ----------
    t0 = time.time()
    inp = F.to_tensor(pil).to(DEVICE)
    with torch.no_grad():
        out = frcnn([inp])[0]
    t = time.time()-t0

    preds = [box.cpu().numpy() for box,score in zip(out["boxes"], out["scores"]) if score>=CONF]
    m = evaluate(np.array(preds), gt, img_area)
    m["time"] = t
    results_summary["faster_rcnn"].append(m)

    # ---------- Mask R-CNN ----------
    t0 = time.time()
    with torch.no_grad():
        out = mrcnn([inp])[0]
    t = time.time()-t0

    preds = [box.cpu().numpy() for box,score in zip(out["boxes"], out["scores"]) if score>=CONF]
    m = evaluate(np.array(preds), gt, img_area)
    m["time"] = t
    results_summary["mask_rcnn"].append(m)

    # ---------- RF-DETR ----------
    t0 = time.time()
    r = rfdetr(pil, conf=CONF)[0]
    t = time.time()-t0

    preds = r.boxes.xyxy.cpu().numpy()
    m = evaluate(preds, gt, img_area)
    m["time"] = t
    results_summary["rfdetr"].append(m)

# ==============================
# FINAL RESULTS
# ==============================

print("\n====== RESULTS (NO SAM) ======\n")

for k,v in results_summary.items():
    p = np.mean([m["precision"] for m in v])
    r = np.mean([m["recall"] for m in v])
    mi = np.mean([m["mean_iou"] for m in v])
    sr = np.mean([m["small_recall"] for m in v])
    t = np.mean([m["time"] for m in v])

    print(f"Model: {k}")
    print(f" Precision: {p:.3f}")
    print(f" Recall:    {r:.3f}")
    print(f" mIoU:      {mi:.3f}")
    print(f" Small-object recall: {sr:.3f}")
    print(f" Avg inference time: {t:.3f} sec")
    print("-"*50)

Running benchmark on sampled_dataset → 50 images

Loading models...

Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to C:\Users\Adithya R/.cache\torch\hub\checkpoints\maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth


100%|██████████| 170M/170M [00:04<00:00, 41.2MB/s] 



Processing 1/50 : 0000047_03500_d_0000095.jpg

0: 640x640 1 person, 1 car, 1 truck, 167.6ms
Speed: 11.8ms preprocess, 167.6ms inference, 3.7ms postprocess per image at shape (1, 3, 640, 640)

Processing 2/50 : 0000068_02648_d_0000007.jpg

0: 640x640 12 persons, 1 car, 63.7ms
Speed: 4.9ms preprocess, 63.7ms inference, 2.5ms postprocess per image at shape (1, 3, 640, 640)

Processing 3/50 : 0000071_07913_d_0000013.jpg

0: 640x640 6 persons, 1 car, 1 umbrella, 1 surfboard, 60.1ms
Speed: 4.7ms preprocess, 60.1ms inference, 1.7ms postprocess per image at shape (1, 3, 640, 640)

Processing 4/50 : 0000197_00851_d_0000148.jpg

0: 640x640 2 persons, 10 cars, 1 truck, 73.0ms
Speed: 5.7ms preprocess, 73.0ms inference, 1.7ms postprocess per image at shape (1, 3, 640, 640)

Processing 5/50 : 0000225_00460_d_0000002.jpg

0: 640x640 1 person, 11 cars, 2 motorcycles, 1 truck, 70.1ms
Speed: 6.7ms preprocess, 70.1ms inference, 2.2ms postprocess per image at shape (1, 3, 640, 640)

Processing 6/50 : 000

In [3]:
import os, time, cv2, torch, numpy as np
from PIL import Image
from ultralytics import YOLO
from torchvision.models.detection import fasterrcnn_resnet50_fpn, maskrcnn_resnet50_fpn
from torchvision.transforms import functional as F

# ==============================
# CONFIGURATION & PATHS
# ==============================
IMG_DIR = r"C:\Users\Adithya R\CV Hackathon\sampled_dataset\sampled_dataset\images"
OUT_DIR = r"C:\Users\Adithya R\CV Hackathon\outputs"
os.makedirs(OUT_DIR, exist_ok=True)

CONF_THRESH = 0.5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ==============================
# HELPER: VISUALIZATION
# ==============================
def draw_results(image, boxes, label, color):
    """Draws bounding boxes and titles on a BGR image."""
    canvas = image.copy()
    for box in boxes:
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, 2)
    # Background for text
    cv2.rectangle(canvas, (10, 10), (350, 70), (0, 0, 0), -1)
    cv2.putText(canvas, label, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 3)
    return canvas

# ==============================
# LOAD MODELS
# ==============================
print(f"Loading models on {DEVICE}...")
# 1 & 2. Torchvision Models
frcnn = fasterrcnn_resnet50_fpn(weights="DEFAULT").to(DEVICE).eval()
mrcnn = maskrcnn_resnet50_fpn(weights="DEFAULT").to(DEVICE).eval()
# 3. RT-DETR (Ultralytics)
rtdetr = YOLO("rtdetr-l.pt") 

# ==============================
# RUN INFERENCE & COMPARE
# ==============================
test_images = [f for f in os.listdir(IMG_DIR) if f.endswith(('.jpg', '.png'))]

for img_name in test_images:
    print(f"Processing: {img_name}")
    img_path = os.path.join(IMG_DIR, img_name)
    
    # Prep inputs
    pil_img = Image.open(img_path).convert("RGB")
    img_bgr = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    torch_img = F.to_tensor(pil_img).to(DEVICE)
    
    # --- 1. Faster R-CNN ---
    with torch.no_grad():
        out = frcnn([torch_img])[0]
    boxes = out['boxes'][out['scores'] >= CONF_THRESH].cpu().numpy()
    viz_fr = draw_results(img_bgr, boxes, "Faster R-CNN", (255, 100, 0)) # Cyan

    # --- 2. Mask R-CNN ---
    with torch.no_grad():
        out = mrcnn([torch_img])[0]
    boxes = out['boxes'][out['scores'] >= CONF_THRESH].cpu().numpy()
    viz_mr = draw_results(img_bgr, boxes, "Mask R-CNN", (0, 255, 100)) # Green

    # --- 3. RT-DETR ---
    res = rtdetr(pil_img, conf=CONF_THRESH, verbose=False)[0]
    boxes = res.boxes.xyxy.cpu().numpy()
    viz_detr = draw_results(img_bgr, boxes, "RT-DETR", (0, 100, 255)) # Orange

    # --- 4. Create Comparison Grid ---
    # Resize for a manageable 2x2 grid (e.g., 1280x720 total)
    h, w = img_bgr.shape[:2]
    gh, gw = 640, 640 # Grid tile size
    
    tile1 = cv2.resize(viz_fr, (gw, gh))
    tile2 = cv2.resize(viz_mr, (gw, gh))
    tile3 = cv2.resize(viz_detr, (gw, gh))
    tile4 = cv2.resize(img_bgr, (gw, gh)) # Original reference
    cv2.putText(tile4, "Original", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)

    top_row = np.hstack((tile1, tile2))
    bottom_row = np.hstack((tile3, tile4))
    final_grid = np.vstack((top_row, bottom_row))

    # Save output
    cv2.imwrite(os.path.join(OUT_DIR, f"compare_{img_name}"), final_grid)

print(f"\nSuccess! Check results in: {OUT_DIR}")

Loading models on cuda...
Processing: 0000047_03500_d_0000095.jpg
Processing: 0000068_02648_d_0000007.jpg
Processing: 0000071_07913_d_0000013.jpg
Processing: 0000197_00851_d_0000148.jpg
Processing: 0000225_00460_d_0000002.jpg
Processing: 0000257_04349_d_0000067.jpg
Processing: 0000288_03801_d_0000803.jpg
Processing: 0000366_01765_d_0000791.jpg
Processing: 9999937_00000_d_0000061.jpg
Processing: 9999940_00000_d_0000037.jpg
Processing: 9999943_00000_d_0000045.jpg
Processing: 9999945_00000_d_0000153.jpg
Processing: 9999948_00000_d_0000001.jpg
Processing: 9999951_00000_d_0000015.jpg
Processing: 9999951_00000_d_0000139.jpg
Processing: 9999955_00000_d_0000121.jpg
Processing: 9999955_00000_d_0000183.jpg
Processing: 9999955_00000_d_0000193.jpg
Processing: 9999955_00000_d_0000275.jpg
Processing: 9999955_00000_d_0000318.jpg
Processing: 9999955_00000_d_0000357.jpg
Processing: 9999955_00000_d_0000374.jpg
Processing: 9999955_00000_d_0000386.jpg
Processing: 9999955_00000_d_0000422.jpg
Processing: 99

In [4]:
import os
import cv2
import numpy as np
from ultralytics import YOLO

# ==============================
# CONFIGURATION
# ==============================
# I've used the paths from your project setup
IMG_DIR = r"C:\Users\Adithya R\CV Hackathon\sampled_dataset\sampled_dataset\images"
OUT_DIR = r"C:\Users\Adithya R\CV Hackathon\outputs\yolo11"
os.makedirs(OUT_DIR, exist_ok=True)

# YOLOv11x is the largest/most accurate; use 'yolo11n.pt' for raw speed
MODEL_VARIANT = "yolo11x.pt" 
CONF_THRESH = 0.5

# ==============================
# EXECUTION
# ==============================
# Load the model
print(f"Loading {MODEL_VARIANT}...")
model = YOLO(MODEL_VARIANT)

# Get all images
image_files = [f for f in os.listdir(IMG_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for img_name in image_files:
    img_path = os.path.join(IMG_DIR, img_name)
    
    # 1. Run Inference
    results = model(img_path, conf=CONF_THRESH, verbose=False)[0]
    
    # 2. Get the Original Image (BGR)
    original_img = results.orig_img.copy()
    
    # 3. Get the Annotated Image (YOLO draws its own boxes)
    annotated_img = results.plot()
    
    # 4. Create Side-by-Side Comparison
    # We add labels so the judges know exactly what they are looking at
    cv2.putText(original_img, "ORIGINAL", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 
                2, (0, 255, 0), 5, cv2.LINE_AA)
    cv2.putText(annotated_img, f"YOLO DETECTION ({MODEL_VARIANT})", (50, 100), 
                cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 5, cv2.LINE_AA)

    # Combine horizontally (hstack)
    # Note: Both images must be the same size. YOLO's results.plot() handles this.
    comparison = np.hstack((original_img, annotated_img))
    
    # 5. Save the result
    output_path = os.path.join(OUT_DIR, f"yolo_comp_{img_name}")
    cv2.imwrite(output_path, comparison)
    print(f"Processed: {img_name}")

print(f"\nDone! View your comparisons here: {OUT_DIR}")

Loading yolo11x.pt...
Processed: 0000047_03500_d_0000095.jpg
Processed: 0000068_02648_d_0000007.jpg
Processed: 0000071_07913_d_0000013.jpg
Processed: 0000197_00851_d_0000148.jpg
Processed: 0000225_00460_d_0000002.jpg
Processed: 0000257_04349_d_0000067.jpg
Processed: 0000288_03801_d_0000803.jpg
Processed: 0000366_01765_d_0000791.jpg
Processed: 9999937_00000_d_0000061.jpg
Processed: 9999940_00000_d_0000037.jpg
Processed: 9999943_00000_d_0000045.jpg
Processed: 9999945_00000_d_0000153.jpg
Processed: 9999948_00000_d_0000001.jpg
Processed: 9999951_00000_d_0000015.jpg
Processed: 9999951_00000_d_0000139.jpg
Processed: 9999955_00000_d_0000121.jpg
Processed: 9999955_00000_d_0000183.jpg
Processed: 9999955_00000_d_0000193.jpg
Processed: 9999955_00000_d_0000275.jpg
Processed: 9999955_00000_d_0000318.jpg
Processed: 9999955_00000_d_0000357.jpg
Processed: 9999955_00000_d_0000374.jpg
Processed: 9999955_00000_d_0000386.jpg
Processed: 9999955_00000_d_0000422.jpg
Processed: 9999956_00000_d_0000056.jpg
Pro

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score

# ---------------- CONFIG ----------------
# Paths to your folders
YOLO_FOLDER = r"C:\Users\Adithya R\CV Hackathon\yolo11"          # Contains yolo_comp_... files
COMPARE_FOLDER = r"C:\Users\Adithya R\CV Hackathon\outputs" # Contains compare_... files

# Thresholds
IOU_THRESHOLD = 0.3
SMALL_AREA = 32 * 32

# ---------------- UTILITIES ----------------
def compute_iou(box1, box2):
    x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    x2, y2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

def extract_model_quadrants(img_path):
    """Crops the 4-way comparison image into quadrants."""
    img = cv2.imread(img_path)
    h, w, _ = img.shape
    q_h, q_w = h // 2, w // 2
    return {
        "Faster R-CNN": img[0:q_h, 0:q_w],
        "Mask R-CNN":   img[0:q_h, q_w:w],
        "RT-DETR":      img[q_h:h, 0:q_w],
        "Original":     img[q_h:h, q_w:w]
    }

def get_yolo_detection(img_path):
    """Crops the 2-way YOLO comparison image."""
    img = cv2.imread(img_path)
    h, w, _ = img.shape
    return img[:, w//2:] # Returns the right side (Detection side)

# ---------------- EVALUATION ENGINE ----------------
def evaluate_models(model_outputs, ground_truth_boxes):
    results = []
    
    for model_name, pred_boxes in model_outputs.items():
        y_true, y_pred = [], []
        ious = []
        matched_gt = set()

        # Match predicted boxes to ground truth
        for pb in pred_boxes:
            best_iou = 0
            best_gt = -1
            for i, gb in enumerate(ground_truth_boxes):
                iou = compute_iou(gb, pb)
                if iou > best_iou:
                    best_iou, best_gt = iou, i
            
            if best_iou >= IOU_THRESHOLD:
                y_true.append(1); y_pred.append(1)
                ious.append(best_iou)
                matched_gt.add(best_gt)
            else:
                y_true.append(0); y_pred.append(1) # False Positive

        # False Negatives
        for i in range(len(ground_truth_boxes)):
            if i not in matched_gt:
                y_true.append(1); y_pred.append(0)

        # Calculate Final Metrics
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec  = recall_score(y_true, y_pred, zero_division=0)
        mAP  = np.mean([1 if i >= 0.5 else 0 for i in ious]) if ious else 0
        
        results.append({
            "Model": model_name,
            "mAP@0.5": mAP,
            "Precision": prec,
            "Recall": rec,
            "Mean IoU": np.mean(ious) if ious else 0
        })
    
    return pd.DataFrame(results)

# ---------------- MAIN EXECUTION ----------------
# Note: In a real scenario, you would loop through your image filenames
# This logic matches the '0000047' and '0000068' tags in your filenames.

# Placeholder ground truth extraction logic
# (Assuming you have a function to get GT for each frame ID)
# all_gt = load_ground_truth_annotations() 

print("Processing and Comparing Models...")
# Example comparison table generation
# final_report = evaluate_models(all_predictions, all_ground_truths)
# print(final_report)

Processing and Comparing Models...


In [8]:
import os
import cv2
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score

# ---------------- CONFIG ----------------
YOLO_DIR = r"C:\Users\Adithya R\CV Hackathon\yolo11"
COMPARE_DIR = r"C:\Users\Adithya R\CV Hackathon\outputs"
IOU_THRESHOLD = 0.3

def get_boxes_by_color(img, color_type):
    if img is None: return []
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    # Adjusted color ranges for better detection from pixels
    if color_type == 'blue': # YOLO / Faster R-CNN
        lower, upper = np.array([100, 100, 50]), np.array([130, 255, 255])
    elif color_type == 'green': # Mask R-CNN
        lower, upper = np.array([40, 70, 70]), np.array([80, 255, 255])
    elif color_type == 'orange': # RT-DETR
        lower, upper = np.array([5, 100, 100]), np.array([25, 255, 255])
    
    mask = cv2.inRange(hsv, lower, upper)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return [cv2.boundingRect(c) for c in contours if cv2.contourArea(c) > 5]

def compute_iou(box1, box2):
    # Convert [x, y, w, h] to [x1, y1, x2, y2]
    b1 = [box1[0], box1[1], box1[0]+box1[2], box1[1]+box1[3]]
    b2 = [box2[0], box2[1], box2[0]+box2[2], box2[1]+box2[3]]
    
    x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = (box1[2]*box1[3]) + (box2[2]*box2[3]) - inter
    return inter / union if union > 0 else 0

def run_comparison():
    all_stats = []
    yolo_files = [f for f in os.listdir(YOLO_DIR) if f.endswith('.jpg')]

    for f_name in yolo_files:
        file_id = f_name.split('_')[2] 
        comp_match = [f for f in os.listdir(COMPARE_DIR) if file_id in f]
        if not comp_match: continue
        
        yolo_full = cv2.imread(os.path.join(YOLO_DIR, f_name))
        comp_full = cv2.imread(os.path.join(COMPARE_DIR, comp_match[0]))
        
        # FIX: YOLO detection is on the RIGHT HALF of the yolo_comp image
        y_h, y_w, _ = yolo_full.shape
        yolo_det_zone = yolo_full[:, y_w//2:]
        
        # FIX: Quadrant splitting for the 4-way comparison
        c_h, c_w, _ = comp_full.shape
        qh, qw = c_h//2, c_w//2
        
        # SET YOLO AS GROUND TRUTH
        gt_boxes = get_boxes_by_color(yolo_det_zone, 'blue')
        if not gt_boxes: continue

        model_preds = {
            "Faster R-CNN": get_boxes_by_color(comp_full[0:qh, 0:qw], 'blue'),
            "Mask R-CNN":   get_boxes_by_color(comp_full[0:qh, qw:c_w], 'green'),
            "RT-DETR":      get_boxes_by_color(comp_full[qh:c_h, 0:qw], 'orange'),
            "YOLO11x (GT)": gt_boxes
        }

        for model, preds in model_preds.items():
            y_true, y_pred, ious = [], [], []
            matched = set()
            
            for pb in preds:
                best_iou, best_gt = 0, -1
                for i, gb in enumerate(gt_boxes):
                    iou = compute_iou(gb, pb)
                    if iou > best_iou: best_iou, best_gt = iou, i
                
                if best_iou >= IOU_THRESHOLD:
                    y_true.append(1); y_pred.append(1); ious.append(best_iou)
                    matched.add(best_gt)
                else:
                    y_true.append(0); y_pred.append(1) # False Positive relative to YOLO
            
            for i in range(len(gt_boxes)):
                if i not in matched: y_true.append(1); y_pred.append(0) # False Negative relative to YOLO
            
            all_stats.append({
                "Model": model,
                "Precision": precision_score(y_true, y_pred, zero_division=0),
                "Recall": recall_score(y_true, y_pred, zero_division=0),
                "Mean IoU": np.mean(ious) if ious else 0
            })

    df = pd.DataFrame(all_stats)
    print("\n--- PERFORMANCE RELATIVE TO YOLO11x ---")
    print(df.groupby("Model").mean())

if __name__ == "__main__":
    run_comparison()


--- PERFORMANCE RELATIVE TO YOLO11x ---
              Precision    Recall  Mean IoU
Model                                      
Faster R-CNN   0.006860  0.004857  0.070609
Mask R-CNN     0.006109  0.006722  0.057378
RT-DETR        0.001445  0.000873  0.014109
YOLO11x (GT)   1.000000  1.000000  1.000000


In [9]:
import os
import cv2
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score

# ---------------- CONFIG ----------------
YOLO_DIR = r"C:\Users\Adithya R\CV Hackathon\yolo11"
COMPARE_DIR = r"C:\Users\Adithya R\CV Hackathon\outputs"
IOU_THRESHOLD = 0.3

def get_boxes_by_color(img, color_type):
    if img is None: return []
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    # Adjusted color ranges for better detection from pixels
    if color_type == 'blue': # YOLO / Faster R-CNN
        lower, upper = np.array([100, 100, 50]), np.array([130, 255, 255])
    elif color_type == 'green': # Mask R-CNN
        lower, upper = np.array([40, 70, 70]), np.array([80, 255, 255])
    elif color_type == 'orange': # RT-DETR
        lower, upper = np.array([5, 100, 100]), np.array([25, 255, 255])
    
    mask = cv2.inRange(hsv, lower, upper)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return [cv2.boundingRect(c) for c in contours if cv2.contourArea(c) > 5]

def compute_iou(box1, box2):
    # Convert [x, y, w, h] to [x1, y1, x2, y2]
    b1 = [box1[0], box1[1], box1[0]+box1[2], box1[1]+box1[3]]
    b2 = [box2[0], box2[1], box2[0]+box2[2], box2[1]+box2[3]]
    
    x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = (box1[2]*box1[3]) + (box2[2]*box2[3]) - inter
    return inter / union if union > 0 else 0

def run_comparison():
    all_stats = []
    yolo_files = [f for f in os.listdir(YOLO_DIR) if f.endswith('.jpg')]

    for f_name in yolo_files:
        file_id = f_name.split('_')[2] 
        comp_match = [f for f in os.listdir(COMPARE_DIR) if file_id in f]
        if not comp_match: continue
        
        yolo_full = cv2.imread(os.path.join(YOLO_DIR, f_name))
        comp_full = cv2.imread(os.path.join(COMPARE_DIR, comp_match[0]))
        if yolo_full is None or comp_full is None: continue

        # --- COORDINATE NORMALIZATION ---
        # 1. Extract and Normalize YOLO (Right side of a 1x2 image)
        y_h, y_w, _ = yolo_full.shape
        yolo_crop = yolo_full[:, y_w//2:]
        yolo_raw_boxes = get_boxes_by_color(yolo_crop, 'blue')
        # Since we cropped the image before getting boxes, 
        # the boxes are already relative to (0,0) of the right half.
        gt_boxes = yolo_raw_boxes 

        # 2. Extract and Normalize Quadrants (2x2 image)
        c_h, c_w, _ = comp_full.shape
        qh, qw = c_h//2, c_w//2

        model_preds = {
            "Faster R-CNN": get_boxes_by_color(comp_full[0:qh, 0:qw], 'blue'),
            "Mask R-CNN":   get_boxes_by_color(comp_full[0:qh, qw:c_w], 'green'),
            "RT-DETR":      get_boxes_by_color(comp_full[qh:c_h, 0:qw], 'orange'),
            "YOLO11x (GT)": gt_boxes
        }

        # --- METRIC CALCULATION ---
        for model, preds in model_preds.items():
            y_true, y_pred, ious = [], [], []
            matched = set()
            
            for pb in preds:
                best_iou, best_gt = 0, -1
                for i, gb in enumerate(gt_boxes):
                    iou = compute_iou(gb, pb)
                    if iou > best_iou: 
                        best_iou, best_gt = iou, i
                
                if best_iou >= IOU_THRESHOLD:
                    y_true.append(1); y_pred.append(1)
                    ious.append(best_iou)
                    matched.add(best_gt)
                else:
                    y_true.append(0); y_pred.append(1) # False Positive
            
            for i in range(len(gt_boxes)):
                if i not in matched: 
                    y_true.append(1); y_pred.append(0) # False Negative
            
            all_stats.append({
                "Model": model,
                "Precision": precision_score(y_true, y_pred, zero_division=0),
                "Recall": recall_score(y_true, y_pred, zero_division=0),
                "Mean IoU": np.mean(ious) if ious else 0
            })

    df = pd.DataFrame(all_stats)
    print("\n--- NORMALIZED PERFORMANCE SUMMARY ---")
    print(df.groupby("Model").mean())

if __name__ == "__main__":
    run_comparison()


--- NORMALIZED PERFORMANCE SUMMARY ---
              Precision    Recall  Mean IoU
Model                                      
Faster R-CNN   0.006860  0.004857  0.070609
Mask R-CNN     0.006109  0.006722  0.057378
RT-DETR        0.001445  0.000873  0.014109
YOLO11x (GT)   1.000000  1.000000  1.000000
